In [ ]:
import numpy as np
import pandas as pd
import os,sys,glob
%load_ext autoreload
%autoreload 2
import plotting_helpers as helper
import nibabel as nib
import scipy
import nilearn
from scipy.spatial.distance import cdist, pdist, squareform
import stats_helpers as sh
import plotting_helpers as ph
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
from nilearn.maskers import NiftiMasker
from nilearn.mass_univariate import permuted_ols
import adult_restmovie_config as ac
import adult_restmovie_utils as au
import infant_restmovie_config as ic 
import infant_restmovie_utils as iu
import hbn_config as hc
import hbn_utils as hu
import narratives_config as nc
import narratives_utils as nu
import partlycloudy_config as pc
import partlycloudy_utils as pu

# Add the text to allow plots to be saved in pdf format with text edit
from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
my_colormap = [ "#fb7fc3","#f6b663", "#faf38f", "#8ad878","#a0c9f0", "#af6adc" ]


In [ ]:
# age in narratives dataset is weirdly formatted
# try to convert to float

def convert_age(age_str):
    # check if nan; return np.nan if so
    if pd.isna(age_str):
        return np.nan
    # check if float, if so, return float*12
    if isinstance(age_str, (int, float)): 
        return age_str * 12
    # otherwise it's a string with , in the middle
    try:
        ages = age_str.split(',')
        ages = [float(age)*12 for age in ages]
        return np.mean(ages)
    except:
        return np.nan

In [ ]:
# Plot participant information for all datasets
pc_df = pd.read_csv('PartlyCloudy/participants.csv', index_col=0)
inf_df = pd.read_csv('infant_restmovie/participant_info.csv', index_col=0) 
hbn_df = pd.read_csv('HBN/basic_cohort_info.csv', index_col=0)
nar_df = pd.read_csv('Narratives/participants.tsv', sep='\t')
adult_df = pd.read_csv('adult_restmovie/adult_rest_participants.csv', index_col=0)

# filter to get only narratives participants who are in the narratives_config file
nar_df = nar_df[nar_df['participant_id'].isin(nc.NARRATIVES_SUBJECTS)]
nar_df['age_months'] = [convert_age(age) for age in nar_df['age']]


# Filter
hbn_df = hbn_df[(hbn_df['cleaning_done']==1)]
inf_df = inf_df[(inf_df['task']!="mickey")]


In [ ]:
hbn_data.head()

In [ ]:
# Prepare data from narratives datasetatasetataset
pc_data = pc_df[['participant_id', 'Age', 'Gender']].copy()
pc_data['dataset'] = 'PartlyCloudy'
pc_data['task'] = 'partly_cloudy'
pc_data['modality'] = 'V'
pc_data['age_months'] = pc_data['Age'] * 12
pc_data['sex'] = pc_data['Gender']
pc_data = pc_data[['participant_id', 'dataset', 'task', 'age_months', 'sex', 'modality']]

inf_data = inf_df[['subject_id', 'age', 'task']].copy()
inf_data['dataset'] = 'InfantRestMovie'
inf_data['age_months'] = inf_data['age']
inf_data['sex'] = None  # sex not in inf_df

inf_data = inf_data.rename(columns={'subject_id': 'participant_id'})
inf_data = inf_data[['participant_id', 'dataset', 'task', 'age_months', 'sex']]
# where task == aeronaut, modality = 'V' ; else modality = 'R'
inf_data['modality'] = inf_data['task'].apply(lambda x: 'V' if x == 'aeronaut' else 'R')

hbn_data = hbn_df[['subject_id', 'age', 'sex']].copy()
hbn_data['dataset'] = 'HBN'
hbn_data['age_months'] = hbn_data['age'] * 12
hbn_data['task'] = 'movieTP'
hbn_data['modality'] = 'AV'
hbn_data = hbn_data.rename(columns={'subject_id': 'participant_id'})
hbn_data = hbn_data[['participant_id', 'dataset', 'task', 'age_months', 'sex']]

# add narratives dataset and adult rest movie dataset to the combined dataframe
nar_data = nar_df[['participant_id', 'age_months', 'sex']].copy()
nar_data['dataset'] = 'Narratives'
nar_data['task'] = 'narratives'
nar_data['modality'] = 'A'

# add adult rest movie dataset
adult_data = adult_df[['ppt', 'age', 'sex']].copy()
adult_data['participant_id'] = adult_data['ppt']
adult_data['dataset'] = 'AdultRestMovie'
adult_data['task'] = 'aeronaut'
adult_data['modality'] = 'V'
adult_data['age_months'] = adult_data['age'] * 12
adult_data = adult_data[['participant_id', 'dataset', 'task', 'age_months', 'sex']]

# Combine all datasets
combined_df = pd.concat([pc_data, inf_data, hbn_data, adult_data, nar_data], ignore_index=True)
combined_df

In [ ]:
combined_df.to_csv('compiled/info/combined_participant_info.csv', index=False)

In [ ]:
# Create ridge plot for age distribution across datasets with separate tasks
fig, axes = plt.subplots(5, 1, figsize=(10, 8), sharex=True)

datasets = ['InfantRestMovie', 'PartlyCloudy', 'HBN', 'AdultRestMovie', 'Narratives']
colors = [my_colormap[0], my_colormap[4], my_colormap[2], my_colormap[1], my_colormap[5]]

for i, (dataset, color) in enumerate(zip(datasets, colors)):
    dataset_data = combined_df[combined_df['dataset'] == dataset]
    tasks = dataset_data['task'].unique()
    
    for j, task in enumerate(tasks):
        if task == 'rest': # Skip rest task if present
            continue
        task_data = dataset_data[dataset_data['task'] == task]['age_months'].dropna()
        
        if len(task_data) > 1:  # Need at least 2 points for KDE
            # Create KDE plot
            task_data.plot.kde(ax=axes[i], color=color, linewidth=2, label=f'{task} (N={len(task_data)})', )
            
            # Fill with different transparency for different tasks
            alpha = 0.3 if j == 0 else 0.6
            axes[i].fill_between(axes[i].get_lines()[-1].get_xdata(), 
                                axes[i].get_lines()[-1].get_ydata(), 
                                alpha=alpha, color=color, edgecolor=color, linewidth=1.5)
    
    # Styling
    axes[i].set_ylabel('Density', fontsize=10)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)
    
    # Add dataset label
    axes[i].text(0.02, 0.7, f'{dataset}', 
                transform=axes[i].transAxes, 
                fontsize=11, verticalalignment='center', fontweight='bold')
    
    # Add legend for tasks
    if len(tasks) > 1:
        axes[i].legend(loc='upper right', fontsize=8, frameon=False)

axes[-1].set_xlabel('Age (months)', fontsize=12)
axes[0].set_title('Age Distribution Across Datasets and Tasks', fontsize=14)
plt.tight_layout()
sns.despine()
# plt.savefig('main_plots/age_distributions_ridge_plot.pdf', format='pdf', bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# Load in all the correlation 
hbn_corrs = pd.read_csv('HBN/results/isc_ide_correlation_null_stats.csv', index_col=0)
pc_corrs = pd.read_csv('PartlyCloudy/results/isc_ide_correlation_null_stats.csv', index_col=0)
inf_corrs = pd.read_csv('infant_restmovie/results/isc_ide_correlation_null_stats.csv', index_col=0)
nar_corrs = pd.read_csv('Narratives/results/ISC_TPHATE_DiffOp_corr.csv').groupby('subject').mean(numeric_only=True).reset_index() # average across runs for each subject
adu_corrs = pd.read_csv('adult_restmovie/results/ISC_TPHATE_DiffOp_corr.csv')
# relable adu_corrs
adu_corrs['subject']= adu_corrs['subject'].apply(lambda x: f'rest_movie_{int(x+1):02d}')

In [ ]:
nar_corrs.head()

In [ ]:
# combine these results datasets into one dataframe with columns dataset, participant_id/subject, zscore, Age, movie_FD, sex, session
combined_corrs = pd.DataFrame(columns=['dataset', 'participant_id', 'zscore', 'age_months', 'FD', 'task', 'modality'])

for index, row in hbn_corrs.iterrows():
    this_row = {'dataset': 'HBN', 'participant_id': row['subject'], 'zscore': row['zscore'], "task":"movieTP",
                                            'FD':row['movie_FD'], 'modality':'AV'}
    # add age from participant info dataframe
    age = combined_df[combined_df['participant_id'] == this_row['participant_id']]['age_months'].values[0]
    this_row['age_months'] = age
    combined_corrs.loc[len(combined_corrs)] = this_row
for index, row in pc_corrs.iterrows():
    this_row = {'dataset': 'PartlyCloudy', 'participant_id': row['subject'], 'zscore': row['zscore'], "task":"partly_cloudy",
                                            'FD':row['movie_FD'],  'modality':'V'}
    # add age from participant info dataframe
    age = combined_df[combined_df['participant_id'] == this_row['participant_id']]['age_months'].values[0]
    this_row['age_months'] = age
    combined_corrs.loc[len(combined_corrs)] = this_row
for index, row in inf_corrs.iterrows():
    this_row = {'dataset': 'InfantRestMovie', 'participant_id': row['subject'], 'zscore': row['zscore'], 
                "task":"aeronaut",  'modality':'V'}
    # add age from participant info dataframe
    age = combined_df[combined_df['participant_id'] == this_row['participant_id']]['age_months'].values[0]
    this_row['age_months'] = age
    combined_corrs.loc[len(combined_corrs)] = this_row
for index, row in nar_corrs.iterrows():
    this_row = {'dataset': 'Narratives', 'participant_id': row['subject'], 'zscore': row['zscore'], "task":"narratives",
                                            'FD':np.nan, 'modality':'A'}
    # add age from participant info dataframe
    age = combined_df[combined_df['participant_id'] == this_row['participant_id']]['age_months'].values[0]
    this_row['age_months'] = age
    combined_corrs.loc[len(combined_corrs)] = this_row
for index, row in adu_corrs.iterrows():
    this_row = {'dataset': 'AdultRestMovie', 'participant_id': row['subject'], 'zscore': row['zscore'], "task":"aeronaut",
                                            'FD':np.nan, 'modality':'V'}
    # add age from participant info dataframe
    age = combined_df[combined_df['participant_id'] == this_row['participant_id']]['age_months'].values[0]
    this_row['age_months'] = age
    combined_corrs.loc[len(combined_corrs)] = this_row



In [ ]:
combined_corrs.to_csv('compiled/results/combined_corrs.csv', index=False)

In [ ]:
combined_corrs = combined_corrs.groupby(['dataset', 'participant_id','modality']).mean(numeric_only=True).reset_index()
# zscore the zscore column within each dataset
combined_corrs['zscore_zscored'] = combined_corrs.groupby('dataset')['zscore'].transform(lambda x: (x - x.mean()) / x.std())
combined_corrs.describe()

In [ ]:
# Run a LMEM with zscore as the dependent variable and age_months, dataset, and their interaction as independent variables
lmem_data = combined_corrs.groupby('dataset').filter(lambda x: len(x) > 10)
# Add random slopes and intercepts for dataset
model1 = smf.mixedlm("zscore ~ age_months", data=lmem_data, groups=lmem_data["dataset"], re_formula="~age_months")
result1 = model1.fit(method="bfgs", reml=False)
aic1 = result1.aic
bic1 = result1.bic

# Now do just a model with random intercepts for dataset, no random slopes
model2 = smf.mixedlm("zscore ~ age_months", data=lmem_data, groups=lmem_data["dataset"]) 
result2 = model2.fit(method="bfgs", reml=False)
aic2 = result2.aic
bic2 = result2.bic

# Now do a model with just fixed effects, no random effects
model3 = smf.ols("zscore ~ age_months", data=lmem_data)
result3 = model3.fit(reml=False)
aic3 = result3.aic
bic3 = result3.bic

print("="*70)
print("LIKELIHOOD-BASED MODEL COMPARISON")
print("="*70)

# Extract log-likelihoods and number of parameters
llf1 = result1.llf
llf2 = result2.llf
llf3 = result3.llf

# Number of parameters
# Model 1: fixed effects (intercept + age_months) + random effects (2 variances + 1 covariance) + residual variance
nparams1 = len(result1.fe_params) + 3 + 1  # 2 fixed + 3 random params + residual var

# Model 2: fixed effects (intercept + age_months) + random intercept variance + residual variance
nparams2 = len(result2.fe_params) + 1 + 1  # 2 fixed + 1 random param + residual var

# Model 3: fixed effects only + residual variance
nparams3 = len(result3.params) + 1  # 2 fixed + residual var

# Test 1: Model 2 vs Model 1 (Are random slopes needed?)
if result1.converged and result2.converged:
    lr_stat_1v2 = -2 * (llf2 - llf1)  # Deviance difference
    df_diff_1v2 = nparams1 - nparams2  # Difference in number of parameters
    p_value_1v2 = stats.chi2.sf(lr_stat_1v2, df_diff_1v2)
    
    print(f"\n1. Model 2 vs Model 1 (Test for Random Slopes):")
    print(f"   H0: Random slopes are not needed (Model 2 is sufficient)")
    print(f"   H1: Random slopes improve fit (Model 1 is better)")
    print(f"   LR χ² = {lr_stat_1v2:.3f}, df = {df_diff_1v2}, p = {p_value_1v2:.4f}")
    
    if p_value_1v2 < 0.05:
        print(f"   → REJECT H0: Random slopes significantly improve fit (use Model 1)")
    else:
        print(f"   → RETAIN H0: Random slopes not needed (use Model 2)")
else:
    print("\n1. Model 2 vs Model 1: CANNOT COMPARE (convergence issues)")

# Test 2: Model 3 vs Model 2 (Are random intercepts needed?)
if result2.converged:
    lr_stat_2v3 = -2 * (llf3 - llf2)  # Deviance difference
    df_diff_2v3 = nparams2 - nparams3  # Difference in number of parameters
    # NOTE: Testing on boundary (variance ≥ 0), so use mixture of chi-squares
    # Conservative: use chi2 with df_diff; more accurate: 0.5*chi2(0) + 0.5*chi2(df_diff)
    p_value_2v3 = 0.5 * stats.chi2.sf(lr_stat_2v3, df_diff_2v3)  # Adjusted for boundary
    
    print(f"\n2. Model 3 vs Model 2 (Test for Random Intercepts):")
    print(f"   H0: Random intercepts are not needed (Model 3 OLS is sufficient)")
    print(f"   H1: Random intercepts improve fit (Model 2 is better)")
    print(f"   LR χ² = {lr_stat_2v3:.3f}, df = {df_diff_2v3}, p = {p_value_2v3:.4f}")
    print(f"   (p-value adjusted for boundary constraint)")
    
    if p_value_2v3 < 0.05:
        print(f"   → REJECT H0: Random intercepts significantly improve fit (use Model 2 or 1)")
    else:
        print(f"   → RETAIN H0: Random intercepts not needed (OLS is sufficient)")
else:
    print("\n2. Model 3 vs Model 2: CANNOT COMPARE (convergence issues)")

# Test 3: Model 3 vs Model 1 (Direct comparison if Model 2 failed)
if result1.converged:
    lr_stat_1v3 = -2 * (llf3 - llf1)
    df_diff_1v3 = nparams1 - nparams3
    p_value_1v3 = 0.5 * stats.chi2.sf(lr_stat_1v3, df_diff_1v3)
    
    print(f"\n3. Model 3 vs Model 1 (Test for All Random Effects):")
    print(f"   H0: No random effects needed (Model 3 OLS is sufficient)")
    print(f"   H1: Random slopes + intercepts improve fit (Model 1 is better)")
    print(f"   LR χ² = {lr_stat_1v3:.3f}, df = {df_diff_1v3}, p = {p_value_1v3:.4f}")
    
    if p_value_1v3 < 0.05:
        print(f"   → REJECT H0: Random effects significantly improve fit (use Model 1)")
    else:
        print(f"   → RETAIN H0: Random effects not needed (OLS is sufficient)")

print("\n" + "="*70)
print("RECOMMENDATION")
print("="*70)

# Determine best model based on available comparisons
if not np.isnan(aic1) and not np.isnan(aic2):
    best_aic = min(aic1, aic2, aic3)
    if best_aic == aic1:
        print("\nBased on AIC: Model 1 (Random Slopes + Intercepts) is best")
    elif best_aic == aic2:
        print("\nBased on AIC: Model 2 (Random Intercepts) is best")
    else:
        print("\nBased on AIC: Model 3 (OLS) is best")
elif not np.isnan(aic2):
    best_aic = min(aic2, aic3)
    if best_aic == aic2:
        print("\nBased on AIC: Model 2 (Random Intercepts) is best")
    else:
        print("\nBased on AIC: Model 3 (OLS) is best")
else:
    print("\nAIC comparison not available due to convergence issues")
    print("Rely on likelihood ratio tests above")

# Calculate variance components if Model 2 converged
if result2.converged:
    print("\n" + "="*70)
    print("VARIANCE COMPONENTS (Model 2)")
    print("="*70)
    
    var_random = result2.cov_re.iloc[0,0]  # Between-dataset variance
    var_residual = result2.scale  # Within-dataset variance
    icc = var_random / (var_random + var_residual)
    
    print(f"\nBetween-dataset variance (random intercepts): {var_random:.4f}")
    print(f"Within-dataset variance (residual): {var_residual:.4f}")
    print(f"Intraclass Correlation (ICC): {icc:.4f}")
    print(f"\nInterpretation: {icc*100:.1f}% of total variance is between datasets")
    
    if icc < 0.05:
        print("  → Very low ICC suggests random effects may not be necessary")
    elif icc < 0.15:
        print("  → Modest clustering by dataset")
    else:
        print("  → Substantial clustering by dataset - random effects are important")

In [ ]:
# Plot zscored zscores by age for each dataset, with a regression line
plt.figure(figsize=(10, 6))
sns.scatterplot(data=combined_corrs, x='age_months', y='zscore', hue='dataset', palette=my_colormap, s=40)
sns.regplot(data=combined_corrs, x='age_months', y='zscore', scatter=False, color='gray', line_kws={'linewidth':1, 'linestyle':'--'})
plt.xlabel('Age (months)', fontsize=12)
plt.ylabel('Z-score', fontsize=12)
plt.title('Z-scores by Age Across Datasets', fontsize=14)
plt.legend(title='Dataset', fontsize=10, title_fontsize=12)
sns.despine()

In [ ]:
# Test model convergense issues wtih different solvers
r1=model1.fit(method="lbfgs", reml=False)
print(r1.aic)
r3=model1.fit(method="cg", reml=False)
print(r3.aic)

In [ ]:
# Run a LMEM with zscore as the dependent variable and age_months, dataset, and their interaction as independent variables
lmem_data = combined_corrs.groupby('dataset').filter(lambda x: len(x) > 10)
# Add random slopes and intercepts for dataset
model1 = smf.mixedlm("zscore_zscored ~ age_months", data=lmem_data, groups=lmem_data["dataset"], re_formula="~age_months")
result1 = model1.fit(method="powell", reml=False)
aic1 = result1.aic
bic1 = result1.bic

# Now do just a model with random intercepts for dataset, no random slopes
model2 = smf.mixedlm("zscore_zscored ~ age_months", data=lmem_data, groups=lmem_data["dataset"]) 
result2 = model2.fit(method="powell", reml=False)
aic2 = result2.aic
bic2 = result2.bic

# Now do a model with just fixed effects, no random effects
model3 = smf.ols("zscore_zscored ~ age_months", data=lmem_data)
result3 = model3.fit(reml=False)
aic3 = result3.aic
bic3 = result3.bic

print("="*70)
print("LIKELIHOOD-BASED MODEL COMPARISON")
print("="*70)

# Extract log-likelihoods and number of parameters
llf1 = result1.llf
llf2 = result2.llf
llf3 = result3.llf

# Number of parameters
# Model 1: fixed effects (intercept + age_months) + random effects (2 variances + 1 covariance) + residual variance
nparams1 = len(result1.fe_params) + 3 + 1  # 2 fixed + 3 random params + residual var

# Model 2: fixed effects (intercept + age_months) + random intercept variance + residual variance
nparams2 = len(result2.fe_params) + 1 + 1  # 2 fixed + 1 random param + residual var

# Model 3: fixed effects only + residual variance
nparams3 = len(result3.params) + 1  # 2 fixed + residual var

# Test 1: Model 2 vs Model 1 (Are random slopes needed?)
if result1.converged and result2.converged:
    lr_stat_1v2 = -2 * (llf2 - llf1)  # Deviance difference
    df_diff_1v2 = nparams1 - nparams2  # Difference in number of parameters
    p_value_1v2 = stats.chi2.sf(lr_stat_1v2, df_diff_1v2)
    
    print(f"\n1. Model 2 vs Model 1 (Test for Random Slopes):")
    print(f"   H0: Random slopes are not needed (Model 2 is sufficient)")
    print(f"   H1: Random slopes improve fit (Model 1 is better)")
    print(f"   LR χ² = {lr_stat_1v2:.3f}, df = {df_diff_1v2}, p = {p_value_1v2:.4f}")
    
    if p_value_1v2 < 0.05:
        print(f"   → REJECT H0: Random slopes significantly improve fit (use Model 1)")
    else:
        print(f"   → RETAIN H0: Random slopes not needed (use Model 2)")
else:
    print("\n1. Model 2 vs Model 1: CANNOT COMPARE (convergence issues)")

# Test 2: Model 3 vs Model 2 (Are random intercepts needed?)
if result2.converged:
    lr_stat_2v3 = -2 * (llf3 - llf2)  # Deviance difference
    df_diff_2v3 = nparams2 - nparams3  # Difference in number of parameters
    # NOTE: Testing on boundary (variance ≥ 0), so use mixture of chi-squares
    # Conservative: use chi2 with df_diff; more accurate: 0.5*chi2(0) + 0.5*chi2(df_diff)
    p_value_2v3 = 0.5 * stats.chi2.sf(lr_stat_2v3, df_diff_2v3)  # Adjusted for boundary
    
    print(f"\n2. Model 3 vs Model 2 (Test for Random Intercepts):")
    print(f"   H0: Random intercepts are not needed (Model 3 OLS is sufficient)")
    print(f"   H1: Random intercepts improve fit (Model 2 is better)")
    print(f"   LR χ² = {lr_stat_2v3:.3f}, df = {df_diff_2v3}, p = {p_value_2v3:.4f}")
    print(f"   (p-value adjusted for boundary constraint)")
    
    if p_value_2v3 < 0.05:
        print(f"   → REJECT H0: Random intercepts significantly improve fit (use Model 2 or 1)")
    else:
        print(f"   → RETAIN H0: Random intercepts not needed (OLS is sufficient)")
else:
    print("\n2. Model 3 vs Model 2: CANNOT COMPARE (convergence issues)")

# Test 3: Model 3 vs Model 1 (Direct comparison if Model 2 failed)
if result1.converged:
    lr_stat_1v3 = -2 * (llf3 - llf1)
    df_diff_1v3 = nparams1 - nparams3
    p_value_1v3 = 0.5 * stats.chi2.sf(lr_stat_1v3, df_diff_1v3)
    
    print(f"\n3. Model 3 vs Model 1 (Test for All Random Effects):")
    print(f"   H0: No random effects needed (Model 3 OLS is sufficient)")
    print(f"   H1: Random slopes + intercepts improve fit (Model 1 is better)")
    print(f"   LR χ² = {lr_stat_1v3:.3f}, df = {df_diff_1v3}, p = {p_value_1v3:.4f}")
    
    if p_value_1v3 < 0.05:
        print(f"   → REJECT H0: Random effects significantly improve fit (use Model 1)")
    else:
        print(f"   → RETAIN H0: Random effects not needed (OLS is sufficient)")

print("\n" + "="*70)
print("RECOMMENDATION")
print("="*70)

# Determine best model based on available comparisons
if not np.isnan(aic1) and not np.isnan(aic2):
    best_aic = min(aic1, aic2, aic3)
    if best_aic == aic1:
        print("\nBased on AIC: Model 1 (Random Slopes + Intercepts) is best")
    elif best_aic == aic2:
        print("\nBased on AIC: Model 2 (Random Intercepts) is best")
    else:
        print("\nBased on AIC: Model 3 (OLS) is best")
elif not np.isnan(aic2):
    best_aic = min(aic2, aic3)
    if best_aic == aic2:
        print("\nBased on AIC: Model 2 (Random Intercepts) is best")
    else:
        print("\nBased on AIC: Model 3 (OLS) is best")
else:
    print("\nAIC comparison not available due to convergence issues")
    print("Rely on likelihood ratio tests above")

# Calculate variance components if Model 2 converged
if result2.converged:
    print("\n" + "="*70)
    print("VARIANCE COMPONENTS (Model 2)")
    print("="*70)
    
    var_random = result2.cov_re.iloc[0,0]  # Between-dataset variance
    var_residual = result2.scale  # Within-dataset variance
    icc = var_random / (var_random + var_residual)
    
    print(f"\nBetween-dataset variance (random intercepts): {var_random:.4f}")
    print(f"Within-dataset variance (residual): {var_residual:.4f}")
    print(f"Intraclass Correlation (ICC): {icc:.4f}")
    print(f"\nInterpretation: {icc*100:.1f}% of total variance is between datasets")
    
    if icc < 0.05:
        print("  → Very low ICC suggests random effects may not be necessary")
    elif icc < 0.15:
        print("  → Modest clustering by dataset")
    else:
        print("  → Substantial clustering by dataset - random effects are important")

In [ ]:
# Plot zscored zscores by age for each dataset, with a regression line
plt.figure(figsize=(10, 6))
sns.scatterplot(data=combined_corrs, x='age_months', y='zscore_zscored', hue='dataset', palette=my_colormap, s=40)
sns.regplot(data=combined_corrs, x='age_months', y='zscore_zscored', scatter=False, color='gray', line_kws={'linewidth':1, 'linestyle':'--'})
plt.xlabel('Age (months)', fontsize=12)
plt.ylabel('Z-scored Z-score', fontsize=12)
plt.title('Z-scored Z-scores by Age Across Datasets', fontsize=14)
plt.legend(title='Dataset', fontsize=10, title_fontsize=12)
sns.despine()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Assume you have your data
# df with columns: z, age_months, dataset

# Model 1: Random intercepts only
model_ri = smf.mixedlm("zscore ~ age_months", 
                       data=combined_corrs, 
                       groups=combined_corrs["dataset"])

try:
    result_ri = model_ri.fit(reml=False)
    print("Random Intercepts Model:")
    print(f"  Converged: {result_ri.converged}")
    print(f"  Log-likelihood: {result_ri.llf}")
    print(f"  AIC: {result_ri.aic}")
    print(f"  BIC: {result_ri.bic}")
except Exception as e:
    print(f"Random Intercepts Model failed: {e}")

# Model 2: Random slopes and intercepts
model_both = smf.mixedlm("zscore ~ age_months", 
                         data=combined_corrs, 
                         groups=combined_corrs["dataset"],
                         re_formula="~age_months")

try:
    result_both = model_both.fit(reml=False)
    print("\nRandom Slopes and Intercepts Model:")
    print(f"  Converged: {result_both.converged}")
    print(f"  Log-likelihood: {result_both.llf}")
    print(f"  AIC: {result_both.aic}")
    print(f"  BIC: {result_both.bic}")
except Exception as e:
    print(f"Random Slopes Model failed: {e}")

# Compare if both converged
if not np.isnan(result_ri.aic) and not np.isnan(result_both.aic):
    print("\nModel Comparison:")
    print(f"  ΔAIC = {result_both.aic - result_ri.aic:.2f}")
    if result_both.aic < result_ri.aic:
        print("  → Random slopes model preferred (lower AIC)")
    else:
        print("  → Random intercepts model preferred (lower AIC)")

In [ ]:
# plot a satterplot of age vs zscore with different colors for each dataset, regression line fit to all data together
# add small jitter to age_months to make points more visible
sns.set_context('talk')
combined_corrs['age_months_jittered'] = combined_corrs['age_months'] + np.random.normal(0, 3, size=len(combined_corrs))

hue_order=['InfantRestMovie', 'AdultRestMovie', 'Narratives','PartlyCloudy', 'HBN', ]

plt.figure(figsize=(8, 8))
sns.scatterplot(data=combined_corrs, x='age_months_jittered', y='zscore', hue='dataset', palette=my_colormap, s=80, alpha=0.7, hue_order=hue_order)
sns.regplot(data=combined_corrs, x='age_months', y='zscore', scatter=False, color='gray', line_kws={'linewidth':2})
# Print on here the variance explained by the fixed effect of age from the LME model
# remove the box around the text and make it white background with some transparency

# Add astericks for significance level of age_months predictor
if result.pvalues['age_months'] < 0.001:
    sig_text = '***'
elif result.pvalues['age_months'] < 0.01:
    sig_text = '**'
elif result.pvalues['age_months'] < 0.05:
    sig_text = '*'
else:
    sig_text = 'n.s.'

# add x axis line at y=0
plt.axhline(0, color='gray', linestyle='--', linewidth=1)

plt.text(0.90, 0.74, f"{sig_text}", transform=plt.gca().transAxes, fontsize=20, verticalalignment='top', bbox=dict( facecolor='white', alpha=1, edgecolor='none' ))

plt.xlabel('Age (months)', fontsize=12)
plt.ylabel('ISC-IDE Correlation Z-Score', fontsize=12)
plt.title('ISC-IDE Correlation vs Age Across Datasets', fontsize=14 )
plt.legend(title='Dataset', fontsize=12, title_fontsize=12, loc='lower right', edgecolor='black')
# add N for each dataset in the legend
handles, labels = plt.gca().get_legend_handles_labels()
these_labels = {'InfantRestMovie':26, 'AdultRestMovie':12, 'Narratives':45,'PartlyCloudy':155, 'HBN':528}
new_labels = []
for label in labels:
    n = len(combined_corrs[combined_corrs['dataset'] == label])
    new_labels.append(f"{label} (N={these_labels[label]})")
plt.legend(handles, new_labels, title='Dataset', fontsize=12, title_fontsize=12, loc='lower right')
# make axis tick
# Add text for variance explained and spearman correlation directly 
rho=r'$\rho$'
# plt.text(0.74, 0.12, f"Marginal R² (age only): {marginal_r2:.2f}\nSpearman\'s {rho}: {scipy.stats.spearmanr(combined_corrs['age_months'], combined_corrs['zscore']).correlation:.2f}",
          
#          transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict( facecolor='white', alpha=1, edgecolor='none' ))
sns.despine()

In [ ]:
# Join all correlation dataframes into a single dataframe
def get_age_group(age):
    if age < 3:
        return '>3'
    elif age < 4:
        return '3-4'
    elif age < 5.3:
        return '4-5'
    elif age < 6:
        return '5-6'
    elif age < 8:
        return '6-8'
    elif age < 10:
        return '8-10'
    elif age < 13:
        return '10-13'
    return 'Adult'

def get_age_group2(age):
    if age < 5:
        return '<5'
    elif age < 6:
        return '5-6'
    elif age < 8.5:
        return '6-8.5'
    elif age < 13:
        return '8.5-13'
    return 'Adult'
    

corrs = pd.DataFrame(columns=['dataset','subject','rho','pval','zscore','group'])
for df, name in zip([pc_corrs, inf_corrs,adu_corrs],
                    ['PartlyCloudy','InfantRestMovie','AdultRestMovie']):
    for row in df.itertuples():
        if name == 'PartlyCloudy':
            age = row.Age
            group = get_age_group2(age)
            dat = 'PartlyCloudy'
        elif name == 'InfantRestMovie':
            group="Infant"
            dat = 'RestMovie'
        elif name == 'AdultRestMovie':
            group="AdultRM"
            dat = 'RestMovie'
        corrs.loc[len(corrs)] = {'dataset':dat,
                              'subject':row.subject,
                              'rho':row.rho,
                              'pval':row.pval,
                              'zscore':row.zscore,
                              'group':group} 

In [ ]:
corrs.head()

In [ ]:
group_order = ['Infant','AdultRM', '3-4', '4-5', '5-6', '6-8', '8-10','10-13','Adult']
palette = ["#EAAEB9", "#E14B67", "#CACDEF","#C1C4EC","#9DA2DA","#636CD0","#2A37BD","#242E92","#060C4D",]
g=sns.barplot(data=corrs, x='group', y='zscore',  palette=palette, order=group_order, alpha=0.8,edgecolor='black', linewidth=1)
sns.stripplot(data=corrs, x='group', y='zscore', palette=palette, size=4, order=group_order,alpha=1,edgecolor='black', linewidth=0.5)
plt.axhline(0, color='black', linestyle='--')
# add statistical significance asterisks
pvals = []
for group in group_order:
    vals = corrs[corrs['group']==group]['zscore'].values
    stat,pval=scipy.stats.ttest_1samp(vals, popmean=0)
    pvals.append(pval)
# Now correct them for multiple comparisons
rej, corrp, _,_ = sh.multipletests(pvals, alpha=0.01, method='bonferroni')
# Add asterisks to the plot
for i, p in enumerate(corrp):
    string = helper.get_asterisks_pvalue(p)
    g.text(i, 14, string, ha='center', va='bottom', color='black', fontsize=14)
g.set(title='ISC-IDE relationship across age groups', xlabel='Dataset and Age Group', ylim=(-6, 16), ylabel='Z-score')
sns.despine()
plt.savefig('main_plots/barplots_isc_ide_correlation_all_datasets_agegroups.pdf', format='pdf', bbox_inches='tight', transparent=True)



In [ ]:
group_order = ['Infant','AdultRM']#, '3-4', '4-5', '5-6', '6-8', '8-10','10-13','Adult']
palette = ["#EAAEB9", "#E14B67"] #, "#CACDEF","#C1C4EC","#9DA2DA","#636CD0","#2A37BD","#242E92","#060C4D",]
fig,ax=plt.subplots(1,1, figsize=(3,4))
g=sns.barplot(data=corrs, x='group', y='zscore',  palette=palette, order=group_order, alpha=0.8,edgecolor='black', linewidth=1,ax=ax)
sns.stripplot(data=corrs, x='group', y='zscore', palette=palette, size=4, order=group_order,alpha=1,edgecolor='black', linewidth=0.5,ax=ax)
plt.axhline(0, color='black', linestyle='--')
# add statistical significance asterisks
pvals = []
for group in group_order:
    vals = corrs[corrs['group']==group]['zscore'].values
    stat,pval=scipy.stats.ttest_1samp(vals, popmean=0)
    pvals.append(pval)
# Now correct them for multiple comparisons
rej, corrp, _,_ = sh.multipletests(pvals, alpha=0.01, method='bonferroni')
# Add asterisks to the plot
for i, p in enumerate(corrp):
    string = helper.get_asterisks_pvalue(p)
    g.text(i, 14, string, ha='center', va='bottom', color='black', fontsize=14)
# Draw a line with ttest significance between the two bars
t,p = scipy.stats.ttest_ind(corrs[corrs['group']=='Infant']['zscore'].values,
                            corrs[corrs['group']=='AdultRM']['zscore'].values)
string = helper.get_asterisks_pvalue(p)
# Draw line
g.plot([0,1],[17,17], color='black')
g.text(0.5, 17, string, ha='center', va='bottom', color='black', fontsize=14)

g.set(title='ISC-IDE relationship', xlabel='', ylim=(-6, 19), ylabel='Z-score',yticks=np.arange(-6,19,6))
sns.despine()
plt.savefig('main_plots/restmovie_barplots_isc_ide_correlation.pdf', format='pdf', bbox_inches='tight', transparent=True)



In [ ]:
2.85 * 10**-3

In [ ]:
group_order = ['<5','5-6', '6-8.5', '8.5-13',  'Adult']#[ '3-4', '4-5', '5-6', '6-8', '8-10','10-13','Adult']
palette = [ "#CACDEF","#9DA2DA","#636CD0","#2A37BD","#242E92"]
fig,ax=plt.subplots(1,1, figsize=(3,4))
g=sns.barplot(data=corrs, x='group', y='zscore',  palette=palette, order=group_order, alpha=0.8,edgecolor='black', linewidth=1,ax=ax)
sns.stripplot(data=corrs, x='group', y='zscore', palette=palette, size=4, order=group_order,alpha=1,edgecolor='black', linewidth=0.5,ax=ax)
plt.axhline(0, color='black', linestyle='--')
# add statistical significance asterisks
pvals = []
for group in group_order:
    vals = corrs[corrs['group']==group]['zscore'].values
    stat,pval=scipy.stats.ttest_1samp(vals, popmean=0)
    pvals.append(pval)
    print(f'Group: {group}, p-value: {pval}, n={len(vals)}')
# Now correct them for multiple comparisons
rej, corrp, _,_ = sh.multipletests(pvals, alpha=0.0001, method='bonferroni')
print(corrp)


In [ ]:
# Add asterisks to the plot
for i, p in enumerate(corrp):
    string = helper.get_asterisks_pvalue(p)
    g.text(i, 11, string, ha='center', va='bottom', color='black', fontsize=14)

# Draw a line with Ftest significance between the first and last bars
pc_data = corrs[corrs['dataset']=='PartlyCloudy']
model = smf.ols('zscore ~ C(group)', data=pc_data).fit()
anova_table = sm.stats.anova_lm(model, typ=2, robust='hc3')
p=anova_table['PR(>F)'][0]
print(anova_table)

g.plot([0,4],[14,14], color='black')
g.text(2, 14, helper.get_asterisks_pvalue(p), ha='center', va='bottom', color='black', fontsize=14)

g.set(title='ISC-IDE relationship', xlabel='', ylim=(-6, 16), ylabel='Z-score',yticks=np.arange(-5,16,5))
sns.despine()
plt.savefig('main_plots/partlycloudy_barplots_isc_ide_correlation.pdf', format='pdf', bbox_inches='tight', transparent=True)



In [ ]:
from scipy import stats

# Fit model with interaction between age_c and dataset
model_interaction = smf.mixedlm('zscore ~ age_c * C(dataset, Treatment(reference="InfantRestMovie"))', data=dev_corrs, groups=dev_corrs['dataset']).fit()
print(model_interaction.summary())

# To specifically test if the interaction with HBN is significant:
# Look at the p-value for age_c:C(dataset, Treatment(reference="InfantRestMovie"))[T.HBN]

# Alternatively, compare models with and without interaction using likelihood ratio test
model_no_interaction = smf.mixedlm('zscore ~ age_c + C(dataset, Treatment(reference="InfantRestMovie"))', data=dev_corrs,groups=dev_corrs['dataset']).fit()
model_no_interaction.summary()

In [ ]:
# C(group, Treatment(reference="Adult"))
lme = smf.mixedlm('zscore ~ 0 + age_c* C(dataset)', data=dev_corrs, groups=dev_corrs['dataset']).fit()
lme.summary()


In [ ]:
ols = smf.ols(formula='zscore ~ age_c', data=dev_corrs).fit()
ols.summary()

In [ ]:
# Create color mapping for datasets
color_map = {
    'InfantRestMovie': my_colormap[0],
    'PartlyCloudy': my_colormap[4],
    'HBN': my_colormap[2]
}

# Fit simple linear regression with age predicting zscore
age_model = smf.ols(formula='zscore ~ age * dataset', data=dev_corrs).fit()

# # Create the plot
# plt.figure(figsize=(8, 6))

# # Plot scatter points colored by dataset
# for dataset in dev_corrs['dataset'].unique():
#     subset = dev_corrs[dev_corrs['dataset'] == dataset]
#     plt.scatter(subset['age'], subset['zscore'], 
#                 color=color_map[dataset], 
#                 alpha=0.9,edgecolors= color_map[dataset], linewidths=1,
#                 label=dataset)

# # Add regression line with 95% CI for all data
# sns.regplot(data=dev_corrs, x='age', y='zscore', 
#             scatter=False, 
#             line_kws={'color': 'black', 'linewidth': 2},
#             ci=95)
# Get regression statistics
rho, p = scipy.stats.spearmanr(dev_corrs['age'], dev_corrs['zscore']) 
beta = age_model.params['age']
pval = age_model.pvalues['age']
ci_lower = age_model.conf_int().loc['age', 0]
ci_upper = age_model.conf_int().loc['age', 1]

# # Add text box with statistics
# textstr = f'ρ = {rho:.03f}\nβ = {beta:.3f}**\n95% CI = [{ci_lower:.3f}, {ci_upper:.3f}]'
# plt.text(0.95, 0.05, textstr, transform=plt.gca().transAxes,
#          fontsize=10, 
#          verticalalignment='bottom',
#          horizontalalignment='right')

# plt.xlabel('Age (months)')
# plt.ylabel('ISC–IDE correlation (zscore)')
# plt.title('Age effects on ISC-IDE relationship, across datasets')
# plt.legend()
# sns.despine()
# plt.tight_layout()
# plt.savefig('main_plots/age_vs_isc_ide_correlation_all_datasets.pdf', format='pdf', bbox_inches='tight', transparent=True)

In [ ]:
age_model.summary()

In [ ]:
# Load average results for aeronaut and mickey tasks
results_dir = iu.get_results_dir()
print(f"Results directory: {results_dir}")

# Define tasks and measures
movie_tasks = ['aeronaut', 'sleep']
measures = {'ISC': 'ISC', 'TPHATE_DiffOp_IDE': 'IDE'}

# Load the average result volumes
avg_results = {}
all_results = {}
avg_imgs = {}
for task in movie_tasks:
    avg_results[task] = {}
    all_results[task] = {}
    avg_imgs[task] = {}
    mask_file = iu.get_intersect_mask(task)
    masker = NiftiMasker(mask_img=mask_file)
    for measure in measures.keys():
        fn = f'{results_dir}/{task}_{measure}_all_subjects_results.nii.gz'
        if os.path.exists(fn):
            img = nib.load(fn)
            # average over subjects - across 4th dimension
            data = masker.fit_transform(img)
            mean_img = np.nanmean(data, axis=0)
            avg_results[task][measure] = mean_img
            mean_nii = masker.inverse_transform(mean_img)
            avg_imgs[task][measure] = mean_nii
            all_results[task][measure] = data
            print(f"Loaded {task} {measure}: {avg_results[task][measure].shape}")
            print(f"Loaded {task} {measure}: {all_results[task][measure].shape}")
            print(f"Loaded {task} {measure}: {avg_imgs[task][measure].shape}")
            
        else:
            print(f"Warning: {fn} not found")


In [ ]:
# Load in average ID for Infant Aeronaut, Adult Aeronaut, Adult Rest, Infant Sleep

# Define tasks and measures
inf_tasks = ['aeronaut', 'sleep']
adu_tasks = ['aeronaut', 'rest']
tasks = {'adults':adu_tasks, 'infants':inf_tasks}
df=[]
# Load the average result volumes
all_results = {}
for group,task in tasks.items():
    all_results[group] = []
    for t in task:
        if group=='adults':
            result_fn = f'adult_restmovie/results/{t}_TPHATE_DiffOp_IDE_all_subject_results.npy'
            results = np.load(result_fn)
        else:
            masker=NiftiMasker(mask_img=iu.get_intersect_mask(t))
            result_fn = f'infant_restmovie/results/{t}_TPHATE_DiffOp_IDE_all_subjects_results.nii.gz'
            nii=nib.load(result_fn)  # just to check it exists
            results=masker.fit_transform(nii)
        r = np.mean(results, axis=1)
        all_results[group].append(r)
        print(f"Loaded {group} {t}: {all_results[group][-1].shape}")
        # add to a pd dataframe
        if t == 'sleep': 
            t='rest'
        temp = pd.DataFrame({'group':group, 'task':t, 'IDE':r})
        df.append(temp)
ide_df = pd.concat(df, axis=0)

In [ ]:
scipy.stats.ttest_ind(adu_corrs['zscore'].values, inf_corrs['zscore'].values)

In [ ]:
np.mean(adu_corrs['zscore'].values),scipy.stats.ttest_1samp(adu_corrs['zscore'].values,0), np.mean(inf_corrs['zscore'].values),scipy.stats.ttest_1samp(inf_corrs['zscore'].values,0)

In [ ]:
helper.get_paired_palette()

In [ ]:
# 1. Test within each group: difference across tasks
print("=" * 60)
print("1. WITHIN-GROUP TESTS: Task effects within each group")
print("=" * 60)

for group in ['adults', 'infants']:
    subset = ide_df[ide_df['group'] == group]
    # Paired t-test (same subjects in both conditions)
    if group == 'adults':
        t, p = scipy.stats.ttest_rel(
            subset[subset['task'] == 'rest']['IDE'], 
            subset[subset['task'] == 'aeronaut']['IDE']
        )
        df = len(subset[subset['task'] == 'rest']) - 1
    else:
        # Independent t-test for infants (different subjects)
        t, p = scipy.stats.ttest_ind(
            subset[subset['task'] == 'rest']['IDE'], 
            subset[subset['task'] == 'aeronaut']['IDE']
        )
        n1 = len(subset[subset['task'] == 'rest'])
        n2 = len(subset[subset['task'] == 'aeronaut'])
        df = n1 + n2 - 2
    print(f"{group.capitalize()}: t({df})={t:.3f}, p={p:.4f}, {helper.get_asterisks_pvalue(p)}")

    # 2. Test main effect of group (collapsing across tasks)
    print("\n" + "=" * 60)
    print("2. MAIN EFFECT OF GROUP (ignoring task)")
    print("=" * 60)
    t, p = scipy.stats.ttest_ind(
        ide_df[ide_df['group'] == 'adults']['IDE'], 
        ide_df[ide_df['group'] == 'infants']['IDE']
    )
    n1 = len(ide_df[ide_df['group'] == 'adults'])
    n2 = len(ide_df[ide_df['group'] == 'infants'])
    df = n1 + n2 - 2
    print(f"Group effect: t({df})={t:.3f}, p={p:.4f} {helper.get_asterisks_pvalue(p)}")

    # 3. Two-way ANOVA: Group × Task interaction
    print("\n" + "=" * 60)
    print("3. TWO-WAY ANOVA: Group × Task interaction")
    print("=" * 60)
    anova_model = smf.ols('IDE ~ C(group) * C(task)', data=ide_df).fit()
    anova_table = sm.stats.anova_lm(anova_model, typ=2)
    print(anova_table)
    print(f"\nDegrees of freedom for each effect:")
    print(f"Group: df = ({int(anova_table.loc['C(group)', 'df'])}, {int(anova_table.loc['Residual', 'df'])})")
    print(f"Task: df = ({int(anova_table.loc['C(task)', 'df'])}, {int(anova_table.loc['Residual', 'df'])})")
    print(f"Group × Task: df = ({int(anova_table.loc['C(group):C(task)', 'df'])}, {int(anova_table.loc['Residual', 'df'])})")

    # Get effect sizes
    print("\n" + "=" * 60)
    print("EFFECT SIZES (Cohen's d)")
    print("=" * 60)
    # Group effect
    d_group = (ide_df[ide_df['group']=='adults']['IDE'].mean() - 
           ide_df[ide_df['group']=='infants']['IDE'].mean()) / ide_df['IDE'].std()
    print(f"Group effect size (d): {d_group:.3f}")

    # Task effect within each group
    for group in ['adults', 'infants']:
        subset = ide_df[ide_df['group'] == group]
        d_task = (subset[subset['task']=='rest']['IDE'].mean() - 
              subset[subset['task']=='aeronaut']['IDE'].mean()) / subset['IDE'].std()
        print(f"Task effect in {group} (d): {d_task:.3f}")

In [ ]:
smf.ols('IDE ~ C(group) + C(task) + C(group)*C(task)', data=ide_df).fit().summary()

In [ ]:
fig,ax = plt.subplots(1,1, figsize=(2, 6))
g=sns.barplot(data=ide_df,ax=ax, x='group', y='IDE', hue='task', palette=helper.get_paired_palette()[4:],alpha=0.6,edgecolor='black', linewidth=2,legend=False)
sns.stripplot(data=ide_df, ax=ax, x='group', y='IDE', hue='task', palette=helper.get_paired_palette()[4:],
              dodge=True,edgecolor='black', linewidth=0.5, legend=False)
g.set(title='Average IDE by Group and Task', ylabel='Average IDE', xlabel='',ylim=(0,28))
# Add statistics annotations showing the comparison between rest and aeronaut within each group and then across groups overall

# Implement this manually
t,p=scipy.stats.ttest_ind(ide_df[ide_df['group']=='adults']['IDE'], ide_df[ide_df['group']=='infants']['IDE'],alternative='two-sided')
y, h, col = 27, 0.2, 'k'
ax.plot([0, 1], [y, y], lw=1.5, c=col)
ax.text(0.5, y + h, helper.get_asterisks_pvalue(p), ha='center', va='bottom', color=col)

temp=ide_df[ide_df['group']=='infants']
t,p=scipy.stats.ttest_ind(temp[temp['task']=='rest']['IDE'], temp[temp['task']=='aeronaut']['IDE'],alternative='two-sided')
y, h, col = 25.5, 0.2, 'k'
ax.plot([0.75, 1.25], [y, y], lw=1.5, c=col)
ax.text(1, y + h, helper.get_asterisks_pvalue(p), ha='center', va='bottom', color=col)


temp=ide_df[ide_df['group']=='adults']
t,p=scipy.stats.ttest_rel(temp[temp['task']=='rest']['IDE'], temp[temp['task']=='aeronaut']['IDE'],alternative='two-sided')
y, h, col = 25.5, 0.2, 'k'
ax.plot([-0.25, 0.25], [y, y], lw=1.5, c=col)
ax.text(0, y + h, helper.get_asterisks_pvalue(p), ha='center', va='bottom', color=col)
sns.despine()
# Move legend outside
# ax.legend(title='Task', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig('main_plots/restmovie_average_IDE_by_group_and_task.pdf', format='pdf', bbox_inches='tight', transparent=True)


In [ ]:
df_inf = pd.read_csv(
    'infant_restmovie/results/parcelwise_results_ISC_IDE.csv', index_col=0
)
# df_adu = pd.read_csv(
#     'adult_restmovie/results/parcelwise_results_ISC_IDE.csv', index_col=0
# )

In [ ]:
df_inf[(df_inf['measure']==
       'ISC') & (df_inf['task']=='aeronaut')].groupby(['region_name']).mean(numeric_only=True).reset_index()['score'].mean()